# FlashCHORD T4 simulation/training smoke test
Short training on NVIDIA's released tissue-box example. This is an integration test;
it does not cover the challenge roster and produces no leaderboard submission.

In [ ]:
import subprocess, sys
from pathlib import Path
subprocess.run([sys.executable,'-m','pip','install','uv'],check=True)
subprocess.run(['uv','python','install','3.11'],check=True)
subprocess.run(['uv','venv','--python','3.11','/tmp/fc-env'],check=True)


In [ ]:
import os
runtime=Path('/tmp/video_to_data')
env=os.environ.copy(); env['GIT_LFS_SKIP_SMUDGE']='1'
subprocess.run(['git','clone','--filter=blob:none','--no-checkout','https://github.com/nvidia-isaac/video_to_data.git',str(runtime)],check=True,env=env)
subprocess.run(['git','sparse-checkout','set','robotic_grounding/flash_chord'],cwd=runtime,check=True,env=env)
subprocess.run(['git','checkout','8c778f905cb35f76f8e6cd6e2ea486da4fb9e0ef'],cwd=runtime,check=True,env=env)
fc=runtime/'robotic_grounding/flash_chord'
py='/tmp/fc-env/bin/python'
subprocess.run(['uv','pip','install','--python',py,'-e',str(fc)+'[train]'],check=True)


In [ ]:
probe = """
import json
from pathlib import Path
import jax, jax.numpy as jnp
import warp as wp
import newton
import flash_chord
wp.init()
a=jnp.arange(256,dtype=jnp.float32)
value=float(jnp.sum(a*a).block_until_ready())
report={'jax_devices':[str(d) for d in jax.devices()],
 'jax_sum_squares':value,'warp_cuda_available':wp.is_cuda_available(),
 'newton_import':True,'flash_chord_import':True}
Path('/kaggle/working/flashchord_runtime_report.json').write_text(json.dumps(report,indent=2))
assert any(d.platform=='gpu' for d in jax.devices())
assert report['warp_cuda_available']
print(json.dumps(report,indent=2))
"""
subprocess.run([py,'-c',probe],check=True)


In [ ]:
import hashlib, shutil, urllib.request
from concurrent.futures import ThreadPoolExecutor
rev='8c778f905cb35f76f8e6cd6e2ea486da4fb9e0ef'
assets_path='robotic_grounding/source/robotic_grounding/robotic_grounding/assets/human_motion_data/ego_recon'
subprocess.run(['git','sparse-checkout','add',assets_path],cwd=runtime,check=True,env=env)
# Materialize only the LFS assets needed for floating Sharpa and this example.
roots=[fc/'src/flash_chord/assets/meshes/sharpawave',runtime/assets_path]
pointers=[]
for base in roots:
 for f in base.rglob('*'):
  if f.is_file() and f.stat().st_size<1024:
   raw=f.read_bytes()
   if raw.startswith(b'version https://git-lfs.github.com/spec/v1'):
    fields=dict(line.split(' ',1) for line in raw.decode().splitlines())
    pointers.append((f,fields['oid'].split(':')[1],int(fields['size'])))
def fetch(item):
 f,digest,size=item
 relative=f.relative_to(runtime).as_posix()
 url='https://media.githubusercontent.com/media/nvidia-isaac/video_to_data/'+rev+'/'+relative
 tmp=f.with_suffix(f.suffix+'.download')
 h=hashlib.sha256()
 with urllib.request.urlopen(url,timeout=180) as response,tmp.open('wb') as out:
  while chunk:=response.read(1024*1024):
   h.update(chunk);out.write(chunk)
 assert tmp.stat().st_size==size and h.hexdigest()==digest, relative
 tmp.replace(f)
with ThreadPoolExecutor(max_workers=4) as pool:list(pool.map(fetch,pointers))
shutil.copytree(runtime/assets_path,fc/'src/flash_chord/assets/human_motion_data/ego_recon',dirs_exist_ok=True)
print('Verified LFS assets:',len(pointers))


In [ ]:
reference=fc/'src/flash_chord/assets/human_motion_data/ego_recon/processed/sequence_id=tissue_box_simple/robot_name=sharpa_wave'
output=Path('/kaggle/working/tissue_smoke')
command=[py,'scripts/train_flash_sac.py','experiment=sharpa_flash_sac',
 f'task.parquet="{reference}"',f'output_dir={output}',
 'scene.world_count=64','training.total_environment_steps=131072',
 'training.replay.capacity=16384','training.replay.minimum_size=1024',
 'training.replay.batch_size=256','training.replay.observation_storage_dtype=float16',
 'training.checkpoint.save_interval_environment_steps=65536',
 'logging.mode=disabled','reset.seed=42']
train_env=env.copy();train_env['XLA_PYTHON_CLIENT_PREALLOCATE']='false'
subprocess.run(command,cwd=fc,env=train_env,check=True)
checkpoints=list(output.glob('policy_*.safetensors'))
assert checkpoints,'Trainer completed without a policy checkpoint'
print('Saved checkpoints:',[str(p) for p in checkpoints])
